# FLUX LoRA Preparation

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**⚠️ Do not run all cells: rename and resize cells permanently modify your images**

## Install

In [ ]:
# Install dependencies
!pip install datasets datasets[vision] --quiet
!pip install transformers qwen-vl-utils -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 68.6 MB/s eta 0:00:00


In [ ]:
# Reset (Crash) Environment, then continue to next cell
import importlib, sys, os
os.kill(os.getpid(), 9)

## Setup

In [ ]:
# Setup
import os
from PIL import Image
import pandas as pd
from datasets import Dataset

## Mount Drive

In [ ]:
# Mount your Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Set Folder

In [ ]:
# Set folder
DATASET_DIR = '/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/dataset_FLUX.1'

In [ ]:
# Check your current dataset filenames
!ls {DATASET_DIR}

bof_aar_lacha_01.png  bof_aar_lacha_11.png  bof_aar_lacha_21.png
bof_aar_lacha_02.png  bof_aar_lacha_12.png  bof_aar_lacha_22.png
bof_aar_lacha_03.png  bof_aar_lacha_13.png  bof_aar_lacha_23.png
bof_aar_lacha_04.png  bof_aar_lacha_14.png  bof_aar_lacha_24.png
bof_aar_lacha_05.png  bof_aar_lacha_15.png  bof_aar_lacha_25.png
bof_aar_lacha_06.png  bof_aar_lacha_16.png  bof_aar_lacha_26.png
bof_aar_lacha_07.png  bof_aar_lacha_17.png  bof_aar_lacha_27.png
bof_aar_lacha_08.png  bof_aar_lacha_18.png  bof_aar_lacha_28.png
bof_aar_lacha_09.png  bof_aar_lacha_19.png  bof_aar_lacha_29.png
bof_aar_lacha_10.png  bof_aar_lacha_20.png  bof_aar_lacha_30.png


## Batch Edit File Names
**⚠️ Run cells below with caution — changes to files are irreversible**

In [ ]:
# *** Running this cell will rename all your image files ***
# Your dataset should be .jpg or .png

FILE_NAME = "bof_aar_lacha"
FILE_TYPE = ".png"

files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith(FILE_TYPE)])

for i, f in enumerate(files, 1):
    old = os.path.join(DATASET_DIR, f)
    new = os.path.join(DATASET_DIR, f"{FILE_NAME}_{i:02d}{FILE_TYPE}")
    os.rename(old, new)
    print(f"{f} → {FILE_NAME}_{i:02d}{FILE_TYPE}")

20260517_205345_part2_00001_.png → bof_aar_lacha_01.png
20260517_205657_part2_00001_.png → bof_aar_lacha_02.png
20260517_210341_part2_00001_.png → bof_aar_lacha_03.png
20260517_210935_part2_00001_.png → bof_aar_lacha_04.png
20260517_211838_part2_00001_.png → bof_aar_lacha_05.png
20260517_212700_part2_00001_.png → bof_aar_lacha_06.png
20260517_212953_part2_00001_.png → bof_aar_lacha_07.png
20260517_221036_part2_00001_.png → bof_aar_lacha_08.png
20260517_221647_part2_00001_.png → bof_aar_lacha_09.png
20260517_221929_part2_00001_.png → bof_aar_lacha_10.png
20260517_222208_part2_00001_.png → bof_aar_lacha_11.png
20260517_222513_part2_00001_.png → bof_aar_lacha_12.png
20260517_223048_part2_00001_.png → bof_aar_lacha_13.png
20260517_224531_part2_00001_.png → bof_aar_lacha_14.png
20260517_224826_part2_00001_.png → bof_aar_lacha_15.png
20260517_225659_part2_00001_.png → bof_aar_lacha_16.png
20260517_230732_part2_00001_.png → bof_aar_lacha_17.png
20260517_231533_part2_00001_.png → bof_aar_lacha

## Batch Edit Image Sizes
**⚠️ Run cells below with caution — changes to files are irreversible**

In [ ]:
# *** Running this cell will resize all images in folder to 1024 x 1024 ***
for f in sorted(os.listdir(DATASET_DIR)):
    if f.endswith(FILE_TYPE):
        path = os.path.join(DATASET_DIR, f)
        img = Image.open(path).resize((1024, 1024), Image.LANCZOS)
        img.save(path)
        print(f"{f} → 1024x1024")

In [ ]:
# *** Running this cell will resize all images so the longest edge is 1024 ***
for f in sorted(os.listdir(DATASET_DIR)):
    if f.endswith(FILE_TYPE):
        path = os.path.join(DATASET_DIR, f)
        img = Image.open(path)
        ratio = 1024 / max(img.size)
        new_size = (int(img.width * ratio), int(img.height * ratio))
        img = img.resize(new_size, Image.LANCZOS)
        img.save(path)
        print(f"{f} → {new_size[0]}x{new_size[1]}")

## Create Captions

In [ ]:
# Setup
!pip install transformers qwen-vl-utils -q

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import os
import torch

In [ ]:
# Load Model
model_name = "Qwen/Qwen2-VL-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
fields = ["subject_building", "materials_surfaces", "geometry_form", "color_light", "setting_context", "camera_angle"]

prompt = """Describe this architectural image using exactly these fields, one per line:
subject_building:
materials_surfaces:
geometry_form:
color_light:
setting_context:
camera_angle:

Be concise. No full sentences. Comma separated descriptors only.
For color_light: name exact colors (e.g. coral red, dusty rose pink, cobalt blue, raw concrete grey). No vague terms like 'vibrant' or 'bright'. Specific hues only."""

In [ ]:
rows = []
files = sorted([f for f in os.listdir(DATASET_DIR) if f.lower().endswith((".jpg", ".png", ".webp"))])

for f in files:
    path = os.path.join(DATASET_DIR, f)
    print(f"Processing {f}...")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": path},
        {"type": "text", "text": prompt}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_new_tokens=256)
    caption = processor.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    row = {"file_name": f}
    for line in caption.splitlines():
        for field in fields:
            if line.lower().startswith(field):
                row[field] = line.split(":", 1)[-1].strip()
    rows.append(row)

print(f"\nDone. {len(rows)} images processed.")

Processing bof_aar_lacha_01.png...
Processing bof_aar_lacha_02.png...
Processing bof_aar_lacha_03.png...
Processing bof_aar_lacha_04.png...
Processing bof_aar_lacha_05.png...
Processing bof_aar_lacha_06.png...
Processing bof_aar_lacha_07.png...
Processing bof_aar_lacha_08.png...
Processing bof_aar_lacha_09.png...
Processing bof_aar_lacha_10.png...
Processing bof_aar_lacha_11.png...
Processing bof_aar_lacha_12.png...
Processing bof_aar_lacha_13.png...
Processing bof_aar_lacha_14.png...
Processing bof_aar_lacha_15.png...
Processing bof_aar_lacha_16.png...
Processing bof_aar_lacha_17.png...
Processing bof_aar_lacha_18.png...
Processing bof_aar_lacha_19.png...
Processing bof_aar_lacha_20.png...
Processing bof_aar_lacha_21.png...
Processing bof_aar_lacha_22.png...
Processing bof_aar_lacha_23.png...
Processing bof_aar_lacha_24.png...
Processing bof_aar_lacha_25.png...
Processing bof_aar_lacha_26.png...
Processing bof_aar_lacha_27.png...
Processing bof_aar_lacha_28.png...
Processing bof_aar_l

In [ ]:
for field in fields:
    print(f"\n=== {field.upper()} ===")
    for row in rows:
        print(row.get(field, "MISSING"))


=== SUBJECT_BUILDING ===
Modernist architecture, geometric forms, layered volumes, materials surfaces: concrete, glass, metal, color_light: Coral red, dusty rose pink, cobalt blue, setting_context: Mediterranean coastal, camera_angle: Elevated, wide shot.
Modernist architecture, geometric forms, clean lines, materials: concrete, glass, metal, color: pink, purple, black, white, setting_context: Poolside, leisure, vacation, luxury, camera_angle: High angle, looking down.
modern castle-like structure, geometric forms, materials surfaces: concrete, glass, metal, color_light: dusty rose pink, coral red, cobalt blue, setting_context: coastal, luxurious, vacation, camera_angle: aerial, high.
modern, geometric, tiered, multi-story, pastel colors, glass windows, concrete base
modern castle-like structure, materials_surfaces: blue stucco, geometry_form: rectangular with angular, pointed elements, color_light: cobalt blue, setting_context: coastal, camera_angle: wide shot.
modern, geometric, con

## Create captions as separate text files for FLUX LoRA Training

In [ ]:
import gspread
from google.colab import auth
from google.auth import default
import os

# Auth
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open sheet
sh = gc.open('01_FLUX_Captions')
ws = sh.worksheet('metadata')

# Get all values
data = ws.get_all_values()

# B1 is header, captions start at B2, filenames in A2+
output_dir = DATASET_DIR
os.makedirs(output_dir, exist_ok=True)

count = 0
for row in data[1:]:  # skip header row
    if len(row) < 2:
        continue
    filename = row[0].strip()  # e.g. bof_aar_lacha_01.png
    caption = row[1].strip()
    if not filename or not caption:
        continue

    # Strip extension and add .txt
    base = os.path.splitext(filename)[0]  # bof_aar_lacha_01
    txt_path = os.path.join(output_dir, base + '.txt')

    with open(txt_path, 'w') as f:
        f.write(caption)
    count += 1

print(f'Saved {count} caption files to {output_dir}')

Saved 30 caption files to /content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/dataset_FLUX.1


## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()